# Week 2 — Day 1: Advanced Prompt Engineering
### CalderR Agentic AI Engineering Internship · Monday 29 June 2026

**Today's stack:** Python · Groq API (`openai/gpt-oss-120b`) · plain `groq` SDK (no LangChain yet — that starts Wednesday)

**Scope (from the Week 2 brief):**
- Theory: Chain-of-Thought (CoT), Tree-of-Thought (ToT), Self-Consistency, Meta-Prompting, Prompt Chaining, Negative Prompting
- Reading distilled: Wei et al. 2022 (CoT), Yao et al. 2023 (ToT), Anthropic Prompt Engineering Guide
- Practice: 10 prompt rewrites (vague → engineered)
- Lab: CoT pipeline — compare with/without CoT on 10 math/logic problems, quantify the delta

**Note on model choice:** `llama-3.1-8b-instant` and `llama-3.3-70b-versatile` — the two models almost every Groq tutorial still references — were deprecated by Groq on **17 June 2026**, twelve days before this internship started. This notebook uses `openai/gpt-oss-120b` (Groq's current recommended replacement, MoE, 128K context, supports `reasoning_effort`). If you copy code from older blog posts/YouTube tutorials this week, expect a 400 model-decommissioned error — swap the model string.

---

## 0. Environment Setup

Dependencies for today. We are *not* installing LangChain yet — Wednesday's session introduces it for tool binding. Today is raw `groq` SDK so you see exactly what's going over the wire before a framework abstracts it.

**Pin versions** — Groq ships SDK updates frequently; floating `groq` can silently change default `temperature`/`reasoning_effort` behavior between sessions.

In [2]:
!pip install "groq==0.32.0" python-dotenv tenacity

  Attempting uninstall: groq
    Found existing installation: groq 1.5.0
    Uninstalling groq-1.5.0:
      Successfully uninstalled groq-1.5.0



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# requirements-day1.txt (pin these — generate with `pip freeze | grep -E "groq|pydantic|python-dotenv"` after install)
# groq==0.32.0
# python-dotenv==1.0.1
# pydantic==2.9.2          # not used for output validation today, but installed for Tuesday continuity
# tenacity==9.0.0          # retry/backoff utility, used in Section 6 negative-prompting demo

!pip install -q --upgrade groq python-dotenv tenacity



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os
import time
import json
from pathlib import Path
from dotenv import load_dotenv
from groq import Groq

# Load GROQ_API_KEY from a .env file in the notebook's working directory.
# NEVER hardcode the key in a cell — if you do, it ends up in this .ipynb's
# JSON on disk and in git history the moment you commit it.
load_dotenv()

API_KEY = os.environ.get("GROQ_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "GROQ_API_KEY not found. Create a `.env` file next to this notebook "
        "with the line: GROQ_API_KEY=gsk_...   and add `.env` to .gitignore."
    )

client = Groq(api_key=API_KEY)

# Current production model (replaces the deprecated llama-3.1-8b-instant /
# llama-3.3-70b-versatile pair as of June 2026). 128K context, MoE, 120B total
# params / ~5.1B active per token — cheaper and faster than the old 70B model.
MODEL = "openai/gpt-oss-120b"

print(f"Client ready. Using model: {MODEL}")


Client ready. Using model: openai/gpt-oss-120b


In [6]:
def ask(prompt: str, system: str | None = None, temperature: float = 0.6,
        max_tokens: int = 1024, reasoning_effort: str | None = None) -> str:
    '''Thin wrapper around chat.completions.create for this notebook's demos.

    reasoning_effort: 'low' | 'medium' | 'high' — only supported by gpt-oss
    models on Groq. Higher effort = more internal reasoning tokens = slower
    and costs more output tokens, but improves multi-step accuracy. We default
    to None (provider default = 'medium') and override per-experiment below.
    '''
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    kwargs = dict(model=MODEL, messages=messages, temperature=temperature,
                  max_completion_tokens=max_tokens)
    if reasoning_effort:
        kwargs["reasoning_effort"] = reasoning_effort

    resp = client.chat.completions.create(**kwargs)
    return resp.choices[0].message.content


## 1. Chain-of-Thought (CoT) Prompting

### 1.1 Theory

**Source:** Wei, Jason, et al. *"Chain-of-Thought Prompting Elicits Reasoning in Large Language Models."* NeurIPS 2022.

**Core claim:** LLMs answer multi-step reasoning problems far more accurately when the prompt induces the model to externalize a sequence of intermediate reasoning steps before producing a final answer, instead of jumping straight to the answer token. The paper demonstrated this primarily via **few-shot CoT** — showing the model 2–8 worked examples where the *answer* in each example is preceded by a written-out reasoning chain — and found large gains on arithmetic, commonsense, and symbolic reasoning benchmarks, with the effect emerging mainly at large model scale (the original paper used PaLM-540B; smaller models showed weaker or no gains).

**Why it works (mechanistic intuition, not proven theory):** a transformer's forward pass has a fixed compute budget per output token. Forcing the model to "show work" turns one hard inference (jump to the final token) into many easier ones (each reasoning step conditions on the previous, already-computed step). The chain effectively gives the model extra serial computation it wouldn't otherwise get to spend.

**Two flavors you'll use interchangeably:**

| Variant | Mechanism | When to use |
|---|---|---|
| **Few-shot CoT** | Show 2+ worked examples with explicit reasoning before the answer | Best raw accuracy; costs prompt tokens; needs hand-written exemplars |
| **Zero-shot CoT** | Append a trigger phrase like *"Let's think step by step"* with no examples | Kojima et al. 2022 follow-up — near few-shot quality on many tasks, zero exemplar engineering cost |

**Failure modes to know (not in the marketing, but you'll hit them):**
- CoT can produce a *plausible-looking* but logically broken chain that still lands on the right answer by coincidence — don't mistake "shows reasoning" for "reasoning is verified."
- On tasks the model is already confident/correct on zero-shot, CoT sometimes *hurts* by giving it more rope to talk itself into an error — it's not a free lunch on every task.
- Reasoning length is not the same as reasoning quality — a long chain padded with restated premises isn't doing more work, just more tokens.

### 1.2 Prompt pattern

In [8]:
# --- Pattern: zero-shot CoT vs direct answer, same problem ---

problem = (
    "A bat and ball together cost $1.10. The bat costs $1.00 more than the "
    "ball. How much does the ball cost?"
)
# Classic CRT (Cognitive Reflection Test) item — intuitive wrong answer is
# $0.10. Correct answer is $0.05. Good canary for "system 1 jump" failures.

print("=== DIRECT (no CoT) ===")
print(ask(problem, temperature=0.0))

print("\n=== ZERO-SHOT CoT ===")
cot_prompt = problem + "\n\nLet's think step by step."
print(ask(cot_prompt, temperature=0.0))


=== DIRECT (no CoT) ===
Let  

- \(b\) = price of the ball (in dollars)  
- \(t\) = price of the bat (in dollars)

We are given two facts:

1. Together they cost $1.10:  
   \[
   b + t = 1.10
   \]

2. The bat costs $1.00 more than the ball:  
   \[
   t = b + 1.00
   \]

Substitute the second equation into the first:

\[
b + (b + 1.00) = 1.10 \\
2b + 1.00 = 1.10
\]

Solve for \(b\):

\[
2b = 1.10 - 1.00 = 0.10 \\
b = \frac{0.10}{2} = 0.05
\]

So the ball costs **$0.05** (5 cents).  

(Consequently, the bat costs \(0.05 + 1.00 = 1.05\) dollars, and together they indeed total $1.10.)

=== ZERO-SHOT CoT ===
**Step‑by‑step solution**

1. **Define the unknowns**  
   Let  
   \[
   \text{price of the ball} = x \text{ dollars}
   \]  
   Then, because the bat costs \$1.00 more than the ball,  

   \[
   \text{price of the bat} = x + 1.00 \text{ dollars}
   \]

2. **Write the equation for the total cost**  
   The combined cost of the bat and the ball is given as \$1.10, so  

   \[
   x \;

here direct also gives the right response.

**What to look for in the output above:** the direct prompt frequently snaps to the intuitive-but-wrong $0.10 answer (the classic CRT trap). The CoT version should set up the algebra explicitly (`bat = ball + 1.00`, `bat + ball = 1.10`) and resolve to $0.05. If both answers come out correct on `gpt-oss-120b`, that's expected — it's a strong reasoning model and this exact problem is heavily represented in training data for *this specific instance*. Re-run with a problem you constructed yourself (not from a textbook) to see the gap more honestly — see the exercise below.

In [12]:
# --- Few-shot CoT exemplar pattern ---
# Two worked examples, then the target question with no answer.
few_shot_cot = '''Q: A juggler can juggle 16 balls. Half are golf balls, and half of the
golf balls are blue. How many blue golf balls are there?
A: There are 16 balls total. Half are golf balls, so 16 / 2 = 8 golf balls.
Half of the golf balls are blue, so 8 / 2 = 4 blue golf balls. The answer is 4.

Q: A robe takes 2 bolts of blue fiber and half that much white fiber. How many
bolts in total does it take?
A: It takes 2 bolts of blue fiber. White fiber is half that amount, so
2 / 2 = 1 bolt of white fiber. In total it takes 2 + 1 = 3 bolts. The answer is 3.

Q: A farmer has 17 sheep. All but 9 die. How many sheep does the farmer have left?
A:'''

print(ask(few_shot_cot, temperature=1, max_tokens=150))


A farmer has 17 sheep. If “all but 9 die,” that means the ones that **don’t** die are 9.  
So the farmer still has **9 sheep** left. The answer is 9.


The third question ("All but 9 die") is a language trap, not an arithmetic one — the answer is simply **9** (the survivors), not `17 - 9 = 8`. This is deliberately included to show that few-shot CoT exemplars teach a *format* (show steps, end with "The answer is X"), not necessarily correct logic for problem types unlike the exemplars. Watch whether the model pattern-matches the subtraction structure of the exemplars instead of parsing the actual sentence.

## 2. Tree-of-Thought (ToT) Prompting

### 2.1 Theory

**Source:** Yao, Shunyu, et al. *"Tree of Thoughts: Deliberate Problem Solving with Large Language Models."* NeurIPS 2023.

**Core claim:** CoT commits to a single linear reasoning path. If an early step is wrong, the whole chain inherits the error with no recovery mechanism — the model never "goes back." ToT generalizes CoT into a search problem over a *tree* of intermediate reasoning states:

1. **Decompose** the problem into discrete "thought" steps (not necessarily one-token-at-a-time — a thought is a coherent partial solution, e.g., one line in a Sudoku, one sentence in an essay plan).
2. **Generate** multiple candidate next-thoughts from the current state (branching factor `b`).
3. **Evaluate** each candidate — the paper uses the LLM itself as a heuristic evaluator, scoring states as "sure / maybe / impossible" or giving a numeric value.
4. **Search** over the tree using BFS or DFS, pruning low-value branches, optionally backtracking when a branch dead-ends.

This trades inference cost (many more LLM calls — one per candidate thought, plus evaluator calls) for solution quality on problems where greedy, single-path reasoning genuinely fails: planning tasks, constraint satisfaction (the paper's flagship example is the "Game of 24" arithmetic puzzle), and creative writing where early structural choices matter.

**CoT vs ToT — when to reach for which:**

| | Chain-of-Thought | Tree-of-Thought |
|---|---|---|
| Structure | Single linear path | Branching tree, evaluated, prunable |
| Cost | 1 generation call | O(b × depth) generation + evaluation calls |
| Backtracking | None | Yes — can abandon a branch |
| Good for | Problems where a correct first step usually leads to a correct chain (most arithmetic, most QA) | Problems with multiple plausible-but-wrong early moves, where lookahead/backtracking matters (planning, puzzles, multi-constraint generation) |
| Practical use today | Default choice; cheap | Reserve for cases where you've *measured* CoT failing due to irrecoverable early mistakes, not by default — the cost multiplier is real |

### 2.2 A lightweight ToT implementation

We won't reimplement the paper's full BFS/DFS framework today (that's a Week-3+ scope item) — instead we build a **minimal 1-level ToT**: generate `b` independent candidate solutions, have the model self-evaluate each, then pick the best. This captures the core "branch + evaluate + select" idea without a multi-level search tree.

In [26]:
def tree_of_thought_solve(problem: str, n_branches: int = 3) -> dict:
    '''Minimal ToT: generate n independent candidate reasoning paths,
    score each with a separate evaluator call, return the best.

    This is a *breadth-1* ToT (one level of branching, no backtracking) —
    enough to demonstrate the branch-then-evaluate pattern without building
    the full recursive search from the paper.
    '''
    candidates = []
    for i in range(n_branches):
        # temperature > 0 is essential here — at temperature=0 all branches
        # would be near-identical and "search" becomes pointless.
        candidate = ask(
            problem + "\n\nThink through this carefully, step by step.",
            temperature=0.8,
            max_tokens=400,
        )
        candidates.append(candidate)

    # Evaluator pass: separate call, asked to be a critic, not a solver.
    eval_prompt = "Problem:\n" + problem + "\n\n"
    for i, c in enumerate(candidates):
        eval_prompt += f"--- Candidate {i+1} ---\n{c}\n\n"
    eval_prompt += (
        "Evaluate each candidate's reasoning for correctness only. "
        "Respond with exactly one line in the form: BEST=<candidate number>, "
        "REASON=<one sentence why the others are wrong or this one is more rigorous>."
    )
    verdict = ask(eval_prompt, temperature=0.0, max_tokens=150)

    return {"candidates": candidates, "verdict": verdict}


In [27]:
# Game-of-24-style problem: classic ToT benchmark from the paper.
# Using the numbers 4, 9, 10, 13 — make 24 using +, -, *, / and each number once.
game_of_24 = "Using the numbers 4, 9, 10, 13 exactly once each, and the operators +, -, *, /, write an expression that evaluates to 24."

result = tree_of_thought_solve(game_of_24, n_branches=3)

for i, c in enumerate(result["candidates"]):
    print(f"--- Candidate {i+1} (truncated) ---")
    print(c[:300], "...\n")

print("=== EVALUATOR VERDICT ===")
print(result["verdict"])


--- Candidate 1 (truncated) ---
 ...

--- Candidate 2 (truncated) ---
**Step 1 – Look for a simple way to split the numbers into two groups whose results can be combined to give 24.**  
A product is often convenient because \(24 = 4 \times 6\).  
If we can make a 4 from some of the numbers and a 6 from the remaining numbers, multiplying them will give the desired resu ...

--- Candidate 3 (truncated) ---
One way to reach 24 is to first form two simple differences and then multiply them.

**Step 1 – Form the first difference**

\[
13-9 = 4
\]

We have used the numbers **13** and **9**.

**Step 2 – Form the second difference**

\[
10-4 = 6
\]

 ...

=== EVALUATOR VERDICT ===



**What's happening structurally:** three independent generation calls (the branches) + one evaluation call = 4 total LLM calls for one answer, versus 1 call for plain CoT. That 4x cost is the central tradeoff — only worth it when single-path reasoning has a measurably higher failure rate on this problem class. Game-of-24 is a good demo case because a single greedy arithmetic path frequently picks a first operation that makes 24 unreachable from the remaining numbers, and there's no recovery without branching.

## 3. Self-Consistency

### 3.1 Theory

**Source:** Wang, Xuezhi, et al. *"Self-Consistency Improves Chain of Thought Reasoning in Language Models."* ICLR 2023.

**Core claim:** instead of sampling one CoT reasoning path and trusting it, sample *many* independent CoT paths at temperature > 0, extract the final answer from each, and take a **majority vote**. The intuition: correct reasoning paths tend to converge on the same answer even when their intermediate steps differ; incorrect paths tend to scatter across different wrong answers. Majority voting amplifies the convergent signal and cancels the noise.

**This is *not* the same as ToT**, even though both involve generating multiple candidates — the distinction matters:

| | Self-Consistency | Tree-of-Thought |
|---|---|---|
| What varies between samples | Same prompt, different random seed/temperature | Can branch on intermediate states, not just the final answer |
| How the best answer is chosen | Statistical majority vote over final answers | LLM-as-evaluator judges intermediate reasoning quality |
| Backtracking | None — independent samples, no interaction | Can prune/backtrack mid-search |
| Cost model | N parallel calls, no evaluator call | N generation calls + evaluator call(s) |
| Best for | Problems with one correct answer per problem, where wrong paths fail in diverse, non-systematic ways | Problems with high branching factor in valid intermediate states |

**When majority voting fails:** if the model has a *systematic* bias (the same wrong reasoning chain dominates across most samples — e.g., a consistent unit-conversion error), self-consistency converges on the confidently wrong answer just as strongly as on a correct one. It corrects for *random* sampling noise, not for *systematic* model error. Don't treat vote-count as a calibrated confidence score.

### 3.2 Implementation

In [ ]:
import re
from collections import Counter

def extract_final_answer(text: str) -> str:
    '''Pulls a final numeric/short answer out of a CoT response.
    Looks for 'answer is X' / 'answer: X' patterns first; falls back to the
    last line. This is intentionally simple — production systems should
    use a structured-output schema instead (Tuesday's topic) rather than
    regex-scraping free text.
    '''
    match = re.search(r"answer\s*(?:is|:)\s*\$?(-?[\d,./]+)", text, re.IGNORECASE)
    if match:
        return match.group(1).strip().rstrip(".")
    # Fallback: last non-empty line, stripped
    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
    return lines[-1] if lines else "UNPARSEABLE"


def self_consistency_solve(problem: str, n_samples: int = 5, temperature: float = 0.8) -> dict:
    '''Samples n independent CoT chains, extracts each final answer,
    returns the majority-vote answer plus the full vote distribution
    for transparency (never hide the vote spread from yourself in dev —
    a 3/5 majority is a much weaker signal than 5/5).
    '''
    cot_prompt = problem + "\n\nLet's think step by step, then state your final numeric answer as 'The answer is X.'"

    answers = []
    chains = []
    for _ in range(n_samples):
        chain = ask(cot_prompt, temperature=temperature, max_tokens=300) 
        chains.append(chain)
        answers.append(extract_final_answer(chain))
        print(f"Sampled chain: {chain[:100]}... Final answer: {answers[-1]}")

    votes = Counter(answers)
    majority_answer, majority_count = votes.most_common(1)[0]

    return {
        "majority_answer": majority_answer,
        "confidence": majority_count / n_samples,
        "vote_distribution": dict(votes),
        "all_chains": chains,
    }


In [31]:
problem = (
    "A train leaves Lahore at 60 km/h heading toward Multan, 300 km away. "
    "At the same moment, a second train leaves Multan at 90 km/h heading "
    "toward Lahore on the same track. How many hours until they meet?"
)

result = self_consistency_solve(problem, n_samples=5, temperature=0.8)


print("Vote distribution:", result["vote_distribution"])
print(f"Majority answer: {result['majority_answer']}  "
      f"(confidence: {result['confidence']:.0%})")


Sampled chain: The answer is 2.... Final answer: 2
Sampled chain: The answer is 2.... Final answer: 2
Sampled chain: **Step‑by‑step reasoning**

1. The distance between Lahore and Multan is 300 km.  
2. The first trai... Final answer: 2
Sampled chain: The answer is 2.... Final answer: 2
Sampled chain: The answer is 2.... Final answer: 2
Vote distribution: {'2': 5}
Majority answer: 2  (confidence: 100%)


**Reading the vote distribution:** if you see something like `{'2': 4, '1.5': 1}`, that's a 4/5 majority — reasonably trustworthy. If you see 5 different answers with no majority, that's a signal the problem (or your extraction regex) needs attention, not a signal to just trust whichever answer happened to come first. The combined relative speed here is 150 km/h closing a 300 km gap → 2 hours is correct; watch for the model occasionally computing relative speed wrong (adding instead of the correct closing-speed logic, or dividing by the wrong speed).

## 4. Meta-Prompting

### 4.1 Theory

**Definition:** meta-prompting is using an LLM to generate, critique, or refine *prompts themselves*, rather than using the LLM to directly solve the end task. The model operates one level removed from the target problem — it reasons about *how to ask*, not the answer.

Three common patterns, in increasing order of how much you let the model drive:

1. **Prompt generation** — give a task description, ask the model to draft an effective prompt for that task (useful for bootstrapping a prompt library quickly).
2. **Prompt critique/refinement** — give the model an existing prompt + its observed failure modes, ask it to rewrite the prompt to close that gap. This is the most practically useful pattern for iterating on production prompts.
3. **Structural meta-prompting** (Zhang et al. 2024's specific "meta-prompting" technique) — a high-level "conductor" model decomposes a complex task into subtasks, dynamically instantiates specialized "expert" prompts for each subtask, then aggregates results. This overlaps with prompt-chaining (Section 5) but is distinguished by the *decomposition itself* being LLM-generated rather than hand-designed by you.

**Why this matters for production work, not just theory:** prompt engineering by hand is a slow human-in-the-loop iteration cycle. Meta-prompting turns "I wrote a prompt, it failed on edge case X, let me manually rewrite it" into a process the model can partially automate — you supply failure examples, it proposes the fix, you verify.

### 4.2 Implementation — prompt critique loop

In [32]:
def critique_and_refine_prompt(original_prompt: str, failure_examples: list[str]) -> str:
    """Meta-prompting pattern: ask the model to rewrite a prompt given
    concrete cases where it failed. The model never sees the original task's
    *answer* here -- it only reasons about prompt design.
    """
    failures_block = chr(10).join(f"- {f}" for f in failure_examples)
    meta_instruction = (
        "You are a prompt engineering expert. Below is a prompt that is "
        "currently used to instruct an LLM, followed by example outputs "
        "where that prompt produced unsatisfactory results.\n\n"
        f"ORIGINAL PROMPT:\n\"\"\"\n{original_prompt}\n\"\"\"\n\n"
        f"OBSERVED FAILURES:\n{failures_block}\n\n"
        "Rewrite the prompt to fix these specific failure modes. Do not "
        "make the prompt generically longer -- every addition must trace "
        "to one of the listed failures. Output ONLY the revised prompt "
        "text, nothing else."
    )
    return ask(meta_instruction, temperature=0.3, max_tokens=400)


In [33]:
original = "Summarize this customer support ticket."

failures = [
    "Summary omitted the customer's account ID, which downstream routing needs.",
    "Summary included the agent's internal apology boilerplate as if it were customer-reported content.",
    "Summary was 3 paragraphs long for a 2-sentence ticket — verbosity not scaled to input length.",
]

refined = critique_and_refine_prompt(original, failures)
print(refined)


Summarize this customer support ticket, **including the customer's account ID and the issue they reported**, but **exclude any agent internal notes, apologies, or boilerplate text**. Keep the summary concise—no more than a brief paragraph or two short sentences, regardless of the ticket length.


**What a good refined prompt looks like:** it should explicitly require account-ID extraction, explicitly exclude agent boilerplate, and constrain length proportionally (e.g., "summary length should not exceed 1.5x the ticket's original length" or "max 2 sentences for tickets under 50 words"). If the rewritten prompt is just the original with "be concise and accurate" appended, the meta-prompting step added no value — that's the failure mode to watch for: generic reinforcement instead of failure-traced specificity.

## 5. Prompt Chaining

### 5.1 Theory

**Definition:** decomposing a complex task into a sequence of smaller prompts, where each step's output becomes the next step's input. Unlike CoT (one model call reasoning internally) or ToT (branching search within one task), prompt chaining is explicit **pipeline decomposition at the orchestration level** — you, the engineer, decide the stages; each stage is its own LLM call (or non-LLM step).

**Why chain instead of one mega-prompt:**
- **Separation of concerns** — each link does one job, is independently testable, and failures are localized (you know *which* step broke).
- **Different parameters per step** — extraction steps want `temperature=0`; a creative drafting step might want `temperature=0.8`. One monolithic prompt forces one temperature for the whole task.
- **Intermediate validation** — you can insert a Pydantic check (Tuesday's topic) or a human-review gate between links, which is impossible inside a single generation.
- **Context window economy** — each link only needs the slice of context relevant to its sub-task, not the entire problem's context stacked into one giant prompt.

**Tradeoff:** more total LLM calls = more latency and cost, and errors can *compound* across links if there's no validation between them — link 3 trusting link 2's malformed output is a classic silent-failure pattern (this is exactly the kind of thing your LeakWatch audit cycles look for in code; the same principle applies to prompt pipelines).

### 5.2 Implementation — a 3-link chain

Task: turn a raw, unstructured incident note into (1) extracted facts → (2) a severity classification → (3) a one-line stakeholder summary. Each link's prompt is scoped to exactly one job.

In [34]:
def chain_link(name: str, prompt: str, temperature: float) -> str:
    '''Executes one link and prints a labeled trace — in production you'd
    replace print() with structured logging (this previews Thursday's
    error-handling material: every link should be independently observable).
    '''
    output = ask(prompt, temperature=temperature, max_tokens=300)
    print(f"--- LINK: {name} (temp={temperature}) ---")
    print(output, "\n")
    return output


raw_note = (
    "ticket from amaima.dev acct, says api returning 500s since ~14:20 utc, "
    "only on POST /upload, GET endpoints fine, started right after the "
    "deploy at 14:15, no error logs visible to customer, blocking their "
    "demo to investors in 2 hours"
)

# Link 1: extraction — pull structured facts, no interpretation yet.
link1_out = chain_link(
    "Fact Extraction",
    f"Extract only factual claims from this incident note as a bulleted "
    f"list. Do not infer severity or cause yet.\n\nNote:\n{raw_note}",
    temperature=0.0,
)

# Link 2: classification — consumes link 1's output, not the raw note.
link2_out = chain_link(
    "Severity Classification",
    f"Given these extracted facts, classify incident severity as one of: "
    f"LOW, MEDIUM, HIGH, CRITICAL. Justify in one sentence.\n\nFacts:\n{link1_out}",
    temperature=0.0,
)

# Link 3: stakeholder summary — consumes link 2's classification + link 1's facts.
link3_out = chain_link(
    "Stakeholder Summary",
    f"Write a single sentence suitable for a Slack #incidents channel, "
    f"combining the severity and the key fact.\n\nSeverity:\n{link2_out}\n\nFacts:\n{link1_out}",
    temperature=0.4,
)

print("=== FINAL CHAIN OUTPUT ===")
print(link3_out)


--- LINK: Fact Extraction (temp=0.0) ---
- The ticket was submitted by the **amaima.dev** account.  
- The API has been returning **HTTP 500** responses since approximately **14:20 UTC**.  
- The 500 errors are occurring **only on the `POST /upload` endpoint**.  
- **GET endpoints are functioning normally** (no 500 errors reported).  
- The problem began **right after a deployment 

--- LINK: Severity Classification (temp=0.0) ---
**Severity: HIGH** – the deployment caused a complete failure (HTTP 500) of the `POST /upload` endpoint, blocking all upload operations for users while other API functions remain available. 

--- LINK: Stakeholder Summary (temp=0.4) ---
**HIGH severity:** Since ~14:20 UTC, the deployment submitted by amaima.dev has caused HTTP 500 errors on the `POST /upload` endpoint—blocking all uploads—while GET endpoints remain functional. 

=== FINAL CHAIN OUTPUT ===
**HIGH severity:** Since ~14:20 UTC, the deployment submitted by amaima.dev has caused HTTP 500 errors on

**Where this would break in production, and why each link matters:** if link 1 misses "investor demo in 2 hours" as a fact, link 2 has no business-impact signal to weigh and will likely under-classify severity from technical symptoms alone (a 500-only-on-POST pattern reads as MEDIUM in isolation). This is the compounding-error risk mentioned above — a validation step after link 1 (even something as simple as "does the fact list mention any deadline/business-impact phrase from the original note?") would catch this class of silent context loss before it propagates.

## 6. Negative Prompting

### 6.1 Theory

**Definition:** explicitly instructing the model what *not* to do, alongside (or instead of) what to do. Borrowed terminology from diffusion image models (where "negative prompts" steer away from unwanted visual attributes), applied to text generation as explicit constraint-setting.

**Why this needs care — negative instructions are weaker than positive ones, structurally:** LLMs are next-token predictors trained overwhelmingly on text that *describes* things, not text that explicitly negates and then doesn't mention them. Telling a model "don't mention competitor pricing" puts the token sequence "competitor pricing" directly in the model's recent context — and recent context is exactly what biases next-token generation, regardless of the negation. This is a well-documented practical failure mode, not a theoretical curiosity: negative instructions frequently get *partially* ignored, especially under sampling temperature > 0.

**The mitigation pattern that works better in practice — reframe negative constraints as positive redirections:**

| Weak (pure negative) | Stronger (positive redirection) |
|---|---|
| "Don't mention competitor pricing" | "Discuss only this product's pricing tier and value proposition" |
| "Don't use markdown formatting" | "Respond as plain, unformatted prose" |
| "Don't apologize" | "State the correction directly, in one sentence" |
| "Don't make up sources" | "If you are not certain of a source, state 'no verified source' explicitly instead of citing one" |

**When pure negative instructions are still necessary (and you should still use them):** hard safety/compliance boundaries, explicit format exclusions where there's no clean positive phrasing, and as a *backstop layer* even after positive reframing — belt-and-suspenders, not either/or. The point isn't "never use negative prompts," it's "don't rely on them as your only or primary mechanism for steering away from something you actually care about."

### 6.2 Implementation — measuring the gap

In [37]:
# Demonstration: pure-negative vs positive-redirection framing on the same constraint.

base_task = "Explain what our SaaS product's pricing model is, for a sales call script."

negative_only = base_task + " Do not mention any competitor by name or compare pricing to competitors."

positive_redirect = base_task + (
    " Focus exclusively on this product's three tiers (Starter, Pro, Enterprise) "
    "and the value justification for each. The script should be self-contained "
    "and require no external comparison to make its case."
)

print("=== NEGATIVE-ONLY FRAMING ===")
print(ask(negative_only, temperature=0.7, max_tokens=400))

print("\n=== POSITIVE-REDIRECTION FRAMING ===")
print(ask(positive_redirect, temperature=0.7, max_tokens=400))


=== NEGATIVE-ONLY FRAMING ===
**Sales Call Script – Explaining Our SaaS Pricing Model**

---

### 1. Opening & Context (30‑45 seconds)

> **Rep:** “Thanks for taking the time to speak with me today, [Prospect Name]. Before we dive into how our platform can solve [specific pain point], I’d like to give you a quick overview of how our pricing works so you can see exactly what you’ll be investing in and what you’ll get at each level.”

---

### 2. Core Pricing Structure (1 minute)

> **Rep:** “Our pricing is built around three simple, transparent tiers that align with the size of your team and the depth of functionality you need. All plans are **subscription‑based, billed annually** (with a discounted monthly option available), and include unlimited access to the core platform, automatic updates, and 24/7 support.”

| Tier | Who It’s Ideal For | Key Features Included | Starting Price* |
|------|-------------------|-----------------------|-----------------|
| **Starter** | Small teams (1‑1

**What to check in the negative-only output specifically:** does it spontaneously name a competitor anyway, or hedge with phrasing like *"unlike other providers"* / *"compared to alternatives on the market"* — a near-miss where it obeys the literal letter (no proper noun named) while still violating the spirit (an implicit comparison frame is present)? That near-miss pattern is the actual practical risk with negative prompting, and it's exactly the kind of thing a single example run can fail to surface — which is why Section 6.3 below scales this to N samples instead of trusting one run.

In [40]:
def measure_constraint_violation(prompt: str, violation_check: callable, n_samples: int = 8) -> float:
    '''Runs a prompt N times at temperature>0 and measures how often a
    violation_check predicate fires on the output. One sample proves nothing
    about a probabilistic system — this is the same statistical-confidence
    logic as Section 3's self-consistency vote, applied to constraint testing
    instead of answer correctness.
    '''
    violations = 0
    for _ in range(n_samples):
        output = ask(prompt, temperature=0.9, max_tokens=200)
        if violation_check(output):
            violations += 1
    return violations / n_samples


def mentions_comparison(text: str) -> bool:
    flags = ["competitor", "other provider", "alternative", "compared to",
             "unlike", "versus", "vs.", "rival"]
    
    print( flags)
    return any(f in text.lower() for f in flags)


neg_violation_rate = measure_constraint_violation(negative_only, mentions_comparison, n_samples=8)
pos_violation_rate = measure_constraint_violation(positive_redirect, mentions_comparison, n_samples=8)

print(f"Negative-only framing  — comparison-language violation rate: {neg_violation_rate:.0%}")
print(f"Positive-redirect framing — comparison-language violation rate: {pos_violation_rate:.0%}")


['competitor', 'other provider', 'alternative', 'compared to', 'unlike', 'versus', 'vs.', 'rival']
['competitor', 'other provider', 'alternative', 'compared to', 'unlike', 'versus', 'vs.', 'rival']
['competitor', 'other provider', 'alternative', 'compared to', 'unlike', 'versus', 'vs.', 'rival']
['competitor', 'other provider', 'alternative', 'compared to', 'unlike', 'versus', 'vs.', 'rival']
['competitor', 'other provider', 'alternative', 'compared to', 'unlike', 'versus', 'vs.', 'rival']
['competitor', 'other provider', 'alternative', 'compared to', 'unlike', 'versus', 'vs.', 'rival']
['competitor', 'other provider', 'alternative', 'compared to', 'unlike', 'versus', 'vs.', 'rival']
['competitor', 'other provider', 'alternative', 'compared to', 'unlike', 'versus', 'vs.', 'rival']
['competitor', 'other provider', 'alternative', 'compared to', 'unlike', 'versus', 'vs.', 'rival']
['competitor', 'other provider', 'alternative', 'compared to', 'unlike', 'versus', 'vs.', 'rival']
['competit

If `neg_violation_rate` comes back meaningfully higher than `pos_violation_rate` across these 8 samples each, that's empirical (if small-N) support for the theory above — not just a textbook claim. If the rates come back similar or `gpt-oss-120b` resists the implicit-comparison trap well in both framings, that's also a real, useful finding: it means this specific constraint isn't where this model's negative-prompting weakness shows up, and you'd need a harder constraint (e.g., format exclusion, or a longer multi-turn conversation where context drift has more room to reintroduce the excluded topic) to surface the gap. Either way — don't take the theory on faith; this is the kind of claim you re-test whenever you adopt a new model, since the weakness is model/training-dependent, not a universal law.

## 7. Practice: 10 Prompt Rewrites

Per Monday's brief: *"Practice 10 prompt rewrites."* Each row below takes a vague, underspecified prompt and applies one or more of today's techniques explicitly, with a one-line note on *which* technique and *why*. This is the written deliverable for the "10 prompt rewrites" requirement — keep this table, it's reusable as a personal prompt-pattern reference going forward.

In [41]:
rewrites = [
    {
        "vague": "Write code to check if a number is prime.",
        "engineered": (
            "Write a Python function `is_prime(n: int) -> bool` using type hints. "
            "Handle edge cases explicitly: n < 2, n == 2, even numbers. Use an "
            "O(sqrt(n)) trial-division approach, not O(n). Include a docstring "
            "with one usage example. Do not use any external libraries."
        ),
        "technique": "Negative + positive constraint stacking — explicit edge "
                     "cases prevent the classic n=0/n=1 off-by-one bug; explicit "
                     "complexity bound prevents a naive O(n) loop.",
    },
    {
        "vague": "Is this code secure?",
        "engineered": (
            "Review this code for the OWASP Top 10 categories specifically. "
            "For each category that applies, state: (1) vulnerable or not, "
            "(2) the exact line number if vulnerable, (3) a one-line fix. "
            "Let's think step by step, checking one category at a time before moving to the next."
        ),
        "technique": "Zero-shot CoT trigger + structured negative scope ('OWASP "
                     "Top 10 specifically' prevents a generic 'looks fine' non-answer).",
    },
    {
        "vague": "Summarize this paper.",
        "engineered": (
            "Summarize this paper in exactly 3 parts: (1) the claimed "
            "contribution from the abstract, in one sentence, (2) the actual "
            "method used, in 2-3 sentences, (3) whether the results section "
            "evidence supports the claim in part 1 as strongly as the abstract "
            "implies, or only a weaker version of it."
        ),
        "technique": "Prompt chaining compressed into one prompt via explicit "
                     "stage labels — separates claim from method from evidence "
                     "instead of producing one blended paragraph.",
    },
    {
        "vague": "Make this error message better.",
        "engineered": (
            "Rewrite this error message so a developer unfamiliar with this "
            "codebase can fix the problem without searching the source. "
            "Do not use generic phrasing like 'an error occurred' or "
            "'something went wrong' — state which input/state caused it and "
            "what a valid value would look like."
        ),
        "technique": "Positive redirection of a negative constraint — bans "
                     "vague phrasing by naming the exact phrases to avoid, "
                     "rather than just saying 'be specific.'",
    },
    {
        "vague": "Solve: what is 17% of 240, then add 30.",
        "engineered": (
            "Solve this step by step, writing out each arithmetic operation "
            "on its own line, then state 'The answer is X.' on the final line: "
            "what is 17% of 240, then add 30."
        ),
        "technique": "Zero-shot CoT with an explicit output-format anchor — "
                     "the 'The answer is X.' suffix makes downstream answer "
                     "extraction (Section 3's regex) reliable.",
    },
    {
        "vague": "Generate 3 ideas for a hackathon project.",
        "engineered": (
            "Generate 3 independent hackathon project ideas for a security-"
            "focused 24-hour event. For each, separately argue why it could "
            "win, then separately argue the strongest reason it could fail to "
            "be finished in 24 hours. Recommend the one with the best "
            "win-potential-to-risk ratio, with justification."
        ),
        "technique": "Minimal ToT pattern — generate independent candidates, "
                     "evaluate each on a explicit axis, then select with stated reasoning.",
    },
    {
        "vague": "Translate this technical doc to plain English.",
        "engineered": (
            "Translate this technical doc to plain English for a non-technical "
            "stakeholder. Do not drop any numeric figures, dates, or named "
            "entities present in the original — preserve every one, even "
            "while simplifying the surrounding language."
        ),
        "technique": "Negative prompting for information-preservation — "
                     "without this, simplification often silently drops "
                     "figures/dates as 'detail,' which is a real failure mode "
                     "with status reports specifically.",
    },
    {
        "vague": "Fix this regex.",
        "engineered": (
            "This regex is intended to validate email addresses but is "
            "rejecting valid ones (example: a valid one it rejects: "
            "'user+tag@sub.domain.co'). Identify which part of the regex "
            "causes this rejection, fix it, and state one other edge case "
            "the fixed regex still might mishandle."
        ),
        "technique": "Concrete failure example embedded in the prompt — "
                     "converts 'fix this' into a falsifiable, checkable claim "
                     "instead of an open-ended request the model can satisfy superficially.",
    },
    {
        "vague": "Write a prompt that gets good code reviews from an LLM.",
        "engineered": (
            "I have this prompt for an LLM code reviewer: 'Review this code "
            "and tell me what's wrong.' It currently produces vague feedback "
            "like 'consider improving error handling' with no line numbers or "
            "concrete fixes. Rewrite the prompt to force specific, actionable "
            "output. Output only the revised prompt."
        ),
        "technique": "Meta-prompting (critique-and-refine pattern) — the model "
                     "reasons about prompt design itself, given a concrete "
                     "observed failure mode, exactly as in Section 4.",
    },
    {
        "vague": "Is this incident bad?",
        "engineered": (
            "Step 1: List only the factual claims in this incident note, no "
            "interpretation. Step 2: Using only the facts from Step 1, classify "
            "severity as LOW/MEDIUM/HIGH/CRITICAL with a one-sentence "
            "justification. Step 3: Write one Slack-ready sentence combining "
            "the severity and the single most decision-relevant fact."
        ),
        "technique": "Explicit prompt chaining within a single prompt — three "
                     "labeled stages prevent the model from jumping straight "
                     "to a severity guess without grounding it in extracted facts.",
    },
]

for i, r in enumerate(rewrites, 1):
    print(f"[{i}] VAGUE: {r['vague']}")
    print(f"    ENGINEERED: {r['engineered']}")
    print(f"    TECHNIQUE: {r['technique']}\n")


[1] VAGUE: Write code to check if a number is prime.
    ENGINEERED: Write a Python function `is_prime(n: int) -> bool` using type hints. Handle edge cases explicitly: n < 2, n == 2, even numbers. Use an O(sqrt(n)) trial-division approach, not O(n). Include a docstring with one usage example. Do not use any external libraries.
    TECHNIQUE: Negative + positive constraint stacking — explicit edge cases prevent the classic n=0/n=1 off-by-one bug; explicit complexity bound prevents a naive O(n) loop.

[2] VAGUE: Is this code secure?
    ENGINEERED: Review this code for the OWASP Top 10 categories specifically. For each category that applies, state: (1) vulnerable or not, (2) the exact line number if vulnerable, (3) a one-line fix. Let's think step by step, checking one category at a time before moving to the next.
    TECHNIQUE: Zero-shot CoT trigger + structured negative scope ('OWASP Top 10 specifically' prevents a generic 'looks fine' non-answer).

[3] VAGUE: Summarize this paper.
   

## 8. Lab 2.0 (Monday Applied Practice): CoT Pipeline — With vs Without CoT

**Brief requirement:** *"Build a CoT prompting pipeline. Compare answers with/without CoT on 10 math and logic problems. Document improvements."*

### 8.1 Design

- 10 problems: a mix of arithmetic word problems (CRT-style, designed to trip an intuitive/System-1 answer) and logic puzzles.
- Each problem run twice: once direct (`temperature=0`, no CoT trigger), once with zero-shot CoT (`temperature=0`, `"Let's think step by step"` appended).
- Each result graded against a hand-verified ground truth (you must know the correct answer independently — never grade an LLM's reasoning using the same LLM as the judge of its own correctness for a ground-truth task like this).
- Output: an accuracy delta table + the actual failure transcripts for any problem where direct-answer got it wrong but CoT got it right (or vice versa — that direction is just as informative).

### 8.2 Problem set + ground truth

In [42]:
# Ground truth is hand-verified — do NOT generate this with an LLM, that
# defeats the purpose of having an independent correctness check.

problems = [
    {
        "id": 1,
        "type": "arithmetic_trap",
        "question": "A bat and ball together cost $1.10. The bat costs $1.00 more than the ball. How much does the ball cost?",
        "ground_truth": "0.05",
    },
    {
        "id": 2,
        "type": "arithmetic_trap",
        "question": "If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?",
        "ground_truth": "5",
    },
    {
        "id": 3,
        "type": "rate_word_problem",
        "question": "A train leaves Lahore at 60 km/h heading toward Multan, 300 km away. At the same moment, a second train leaves Multan at 90 km/h heading toward Lahore on the same track. How many hours until they meet?",
        "ground_truth": "2",
    },
    {
        "id": 4,
        "type": "arithmetic_trap",
        "question": "In a lake, there is a patch of lily pads. Every day, the patch doubles in size. If it takes 48 days for the patch to cover the entire lake, how many days would it take for the patch to cover half the lake?",
        "ground_truth": "47",
    },
    {
        "id": 5,
        "type": "logic_puzzle",
        "question": "Three boxes are labeled 'Apples', 'Oranges', and 'Apples and Oranges'. All three labels are wrong. You may draw exactly one fruit from exactly one box to determine the correct label for all three boxes. Which box do you draw from, and what does drawing one orange from it tell you?",
        "ground_truth": "Apples and Oranges box; since all labels are wrong, this box can't contain a mix, so it's pure apples or pure oranges. Drawing an orange means this box is actually pure Oranges, the box labeled 'Apples' must be Apples-and-Oranges (since it can't be Apples, and can't be Oranges which we just found), and the box labeled 'Oranges' must be Apples.",
    },
    {
        "id": 6,
        "type": "arithmetic_multistep",
        "question": "A shop sells a jacket for $80 after a 20% discount from the original price. What was the original price?",
        "ground_truth": "100",
    },
    {
        "id": 7,
        "type": "logic_puzzle",
        "question": "A man is looking at a photo. Someone asks who it is. He says, 'I have no brothers or sisters, but that man's father is my father's son.' Who is in the photo?",
        "ground_truth": "His son",
    },
    {
        "id": 8,
        "type": "arithmetic_trap",
        "question": "All but 9 of a farmer's 17 sheep die. How many sheep does the farmer have left?",
        "ground_truth": "9",
    },
    {
        "id": 9,
        "type": "rate_word_problem",
        "question": "A tank is filled by pipe A in 6 hours and emptied by pipe B in 10 hours. If both pipes are open at the same time on an empty tank, how many hours until the tank is full?",
        "ground_truth": "15",
    },
    {
        "id": 10,
        "type": "logic_puzzle",
        "question": "You have two ropes, each of which takes exactly 60 minutes to burn completely, but they burn unevenly (not at a constant rate along their length). Using only these two ropes and a lighter, how do you measure exactly 45 minutes?",
        "ground_truth": "Light rope A at both ends and rope B at one end simultaneously. Rope A burns out after 30 minutes (since two flames consuming it from both ends meet in half the total burn time). At that moment, light the second end of rope B. Rope B had 30 minutes of burn time left (since 30 of its 60 minutes elapsed from one end); lighting the other end now makes the two flames meet in 15 more minutes. Total: 30 + 15 = 45 minutes.",
    },
]

print(f"Loaded {len(problems)} problems.")


Loaded 10 problems.


In [50]:
def run_direct(question: str) -> str:
    uff = ask(question, temperature=0.0, max_tokens=500)
    print(f"Direct answer: {uff}")
    return uff

def run_cot(question: str) -> str:
    uff = ask(question + "\n\nLet's think step by step.", temperature=0.0, max_tokens=500)
    print(f"CoT answer: {uff}")
    return uff


In [51]:
results = []

for p in problems:
    direct_answer = run_direct(p["question"])
    cot_answer = run_cot(p["question"])
    results.append({
        "id": p["id"],
        "type": p["type"],
        "question": p["question"],
        "ground_truth": p["ground_truth"],
        "direct_answer": direct_answer,
        "cot_answer": cot_answer,
    })
    print(f"Problem {p['id']} ({p['type']}) — done.")


Direct answer: The ball costs **$0.05**.

**Explanation**

Let  

- \(x\) = price of the ball (in dollars)  
- \(x + 1.00\) = price of the bat (the bat is $1.00 more expensive than the ball)

The total cost is given as $1.10:

\[
x + (x + 1.00) = 1.10
\]

Combine like terms:

\[
2x + 1.00 = 1.10
\]

Subtract $1.00 from both sides:

\[
2x = 0.10
\]

Divide by 2:

\[
x = 0.05
\]

So the ball costs **5 cents** and the bat costs \(0.05 + 1.00 = 1.05\) dollars, which together indeed total $1.10.
CoT answer: **Step‑by‑step solution**

1. **Define the unknowns**  
   Let  
   \[
   \text{price of the ball} = x \text{ dollars}
   \]  
   Then, because the bat costs \$1.00 more than the ball,  

   \[
   \text{price of the bat} = x + 1.00 \text{ dollars}
   \]

2. **Write the equation for the total cost**  
   The combined cost of the bat and the ball is given as \$1.10, so  

   \[
   x \;+\; (x + 1.00) \;=\; 1.10
   \]

3. **Simplify the equation**  

   \[
   2x + 1.00 = 1.10
   \]

4. **Sol

### 8.3 Grading

These ground-truth answers are short enough to grade by **eye** for the arithmetic problems (does the response contain the right number?) but the logic puzzles require judgment (does the reasoning identify the correct mechanism, even if phrased differently from the ground truth?). We grade manually below rather than auto-grading with another LLM call — for a 10-problem set, human verification is fast and removes a second LLM's potential correlated blind spots from the loop entirely.

**Action for you:** run the cell above, read each `direct_answer` / `cot_answer` against `ground_truth`, then fill in the `manual_grades` dict below with `True`/`False` for each problem × method. The structure is pre-built — you're filling in judgment calls, not writing new code.

In [52]:
# Fill these in after reading the actual outputs from the cell above.
# Format: {problem_id: (direct_correct: bool, cot_correct: bool)}
manual_grades = {
    1: (True, True),   # bat-and-ball trap
    2: (True, True),   # machines/widgets trap
    3: (True, True),   # closing-speed rate problem
    4: (True, True),   # lily pad doubling trap
    5: (True, True),   # box-labeling logic puzzle
    6: (True, True),   # reverse-discount arithmetic
    7: (True, True),   # family-relation logic puzzle
    8: (True, True),   # "all but 9" language trap
    9: (True, True),   # fill/drain rate problem
    10: (True, True),  # rope-burning logic puzzle
}

# Replace each None with True/False once you've read the transcripts above.
# Example for problem 1, if direct was wrong and CoT was right:
#   1: (False, True),


In [53]:
def summarize_grades(grades: dict) -> None:
    '''Run this AFTER filling in manual_grades above. Computes the accuracy
    delta and flags every problem where the two methods disagreed — those
    disagreement cases are the most informative transcripts to keep for
    Friday's standup ('document improvements' from the brief).
    '''
    ungraded = [pid for pid, (d, c) in grades.items() if d is None or c is None]
    if ungraded:
        print(f"Still ungraded: problem IDs {ungraded}. Fill in manual_grades first.")
        return

    n = len(grades)
    direct_correct = sum(1 for d, _ in grades.values() if d)
    cot_correct = sum(1 for _, c in grades.values() if c)

    print(f"Direct accuracy:  {direct_correct}/{n}  ({direct_correct/n:.0%})")
    print(f"CoT accuracy:     {cot_correct}/{n}  ({cot_correct/n:.0%})")
    print(f"Delta:            {(cot_correct - direct_correct)/n:+.0%}\n")

    print("Disagreement cases (where the two methods differed — keep these for standup):")
    for pid, (d, c) in grades.items():
        if d != c:
            outcome = "CoT helped" if c and not d else "CoT hurt"
            print(f"  Problem {pid}: direct={d}, cot={c}  →  {outcome}")

summarize_grades(manual_grades)


Direct accuracy:  10/10  (100%)
CoT accuracy:     10/10  (100%)
Delta:            +0%

Disagreement cases (where the two methods differed — keep these for standup):


## 9. Self-Check Before Friday

Quick recap — can you answer these without scrolling back up? (These map directly to the Weekly Assessment conceptual questions in the brief.)

1. What's the structural difference between self-consistency and ToT, given that both sample multiple candidates? *(Answer is in Section 3.1's comparison table.)*
2. Why does a pure negative instruction ("don't mention X") structurally risk being weaker than a positive redirection? *(Section 6.1 — it's about what tokens end up in recent context, not just intent.)*
3. In your own 10-problem lab run: did CoT *ever* underperform direct answering? If you observed zero such cases, that's worth questioning — re-read Section 1.1's failure-modes note before assuming CoT is strictly dominant for every problem type.

This notebook intentionally stops before structured outputs / Pydantic validation — that's tomorrow (Tuesday). Today's outputs were graded by eye and parsed with a regex precisely *because* we haven't introduced schema validation yet; don't be surprised that Section 3's `extract_final_answer` feels fragile — fixing that fragility with `with_structured_output()` is the explicit subject of Day 2.
